In [ ]:
# FSDL Lab 8 | Model Monitoring


!pip install gradio --quiet

import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from PIL import Image
import numpy as np
import gradio as gr
import json
import os
from datetime import datetime

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print(f"Gradio: {gr.__version__}")

Device: cuda
Gradio: 5.50.0


In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
    def forward(self, x):
        return self.block(x)

class EMNISTClassifier(nn.Module):
    def __init__(self, num_classes=26):
        super().__init__()
        self.features = nn.Sequential(
            ConvBlock(1, 32),
            ConvBlock(32, 64),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64*7*7, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes),
        )
    def forward(self, x):
        return self.classifier(self.features(x))

model = EMNISTClassifier().to(device)

# train quick if no saved weights
if not os.path.exists("emnist_model.pth"):
    print("Training model...")
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.1307,), (0.3081,))
    ])
    train_dataset = torchvision.datasets.EMNIST(
        root='./data', split='letters',
        train=True, download=True, transform=transform
    )
    train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=2)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()
    model.train()
    for epoch in range(3):
        for x, y in train_loader:
            x, y = x.to(device), (y-1).to(device)
            optimizer.zero_grad()
            loss = criterion(model(x), y)
            loss.backward()
            optimizer.step()
        print(f"Epoch {epoch+1}/3 done")
    torch.save(model.state_dict(), "emnist_model.pth")
else:
    model.load_state_dict(torch.load("emnist_model.pth", map_location=device))
    print("Model loaded from saved weights.")

model.eval()
print("Model ready.")

Training model...


100%|██████████| 562M/562M [00:07<00:00, 74.3MB/s]


Epoch 1/3 done
Epoch 2/3 done
Epoch 3/3 done
Model ready.


In [ ]:
LOG_FILE = "prediction_log.jsonl"

transform_infer = transforms.Compose([
    transforms.Grayscale(),
    transforms.Resize((28, 28)),
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

def log_prediction(image_id, prediction, confidence, flagged=False, flag_reason=""):
    entry = {
        "timestamp": datetime.now().isoformat(),
        "image_id": image_id,
        "prediction": prediction,
        "confidence": round(confidence, 4),
        "flagged": flagged,
        "flag_reason": flag_reason
    }
    with open(LOG_FILE, "a") as f:
        f.write(json.dumps(entry) + "\n")

def predict_letter(image):
    if image is None:
        return "No image provided", ""

    img = Image.fromarray(image.astype(np.uint8))
    img_tensor = transform_infer(img).unsqueeze(0).to(device)

    with torch.no_grad():
        logits = model(img_tensor)
        probs = torch.softmax(logits, dim=1)[0]
        top3_probs, top3_idx = torch.topk(probs, 3)

    result = {}
    for prob, idx in zip(top3_probs, top3_idx):
        letter = chr(65 + idx.item())
        result[letter] = float(prob)

    top_letter = chr(65 + top3_idx[0].item())
    top_conf = float(top3_probs[0])

    # log every prediction automatically
    image_id = f"img_{datetime.now().strftime('%H%M%S%f')}"
    log_prediction(image_id, top_letter, top_conf)

    return result, image_id

def flag_prediction(image_id, flag_reason):
    if not image_id:
        return "No prediction to flag."
    log_prediction(image_id, "", 0.0, flagged=True, flag_reason=flag_reason)
    return f"Flagged! Reason: '{flag_reason}' logged for review."

print("Functions defined.")

Functions defined.


In [ ]:
with gr.Blocks(title="EMNIST Letter Classifier + Monitoring") as app:
    gr.Markdown("# ✍️ Handwritten Letter Classifier")
    gr.Markdown("Draw or upload a letter. Flag wrong predictions to help improve the model.")

    image_id_state = gr.State("")

    with gr.Row():
        with gr.Column():
            image_input = gr.Image(
                label="Draw or upload a letter",
                type="numpy",
                height=280
            )
            predict_btn = gr.Button("Predict", variant="primary")

        with gr.Column():
            label_output = gr.Label(
                label="Top 3 Predictions",
                num_top_classes=3
            )

    with gr.Row():
        with gr.Column():
            flag_reason = gr.Textbox(
                label="Flag this prediction",
                placeholder="e.g. 'wrong prediction, it was B' or 'image was unclear'"
            )
            flag_btn = gr.Button("🚩 Flag Wrong Prediction", variant="stop")
            flag_status = gr.Textbox(label="Flag Status", interactive=False)

    predict_btn.click(
        fn=predict_letter,
        inputs=image_input,
        outputs=[label_output, image_id_state]
    )

    flag_btn.click(
        fn=flag_prediction,
        inputs=[image_id_state, flag_reason],
        outputs=flag_status
    )

    gr.Markdown("### How monitoring works")
    gr.Markdown(
        "Every prediction is automatically logged to `prediction_log.jsonl`. "
        "When you flag a wrong prediction, it gets marked in the log with your reason. "
        "In production, this log feeds into a retraining pipeline."
    )

app.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://88e11f2f255c50060d.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
# after using the app for a bit, analyze what got logged
import pandas as pd

if os.path.exists(LOG_FILE):
    logs = []
    with open(LOG_FILE, "r") as f:
        for line in f:
            logs.append(json.loads(line))

    df = pd.DataFrame(logs)
    print(f"Total predictions logged: {len(df)}")
    print(f"Flagged predictions: {df['flagged'].sum()}")
    print(f"\nMost common predictions:")
    print(df[df['prediction'] != '']['prediction'].value_counts())
    print(f"\nFlagged entries:")
    print(df[df['flagged'] == True][['timestamp', 'prediction', 'flag_reason']])
else:
    print("No logs yet — run the app and make some predictions first.")


Total predictions logged: 2
Flagged predictions: 1

Most common predictions:
prediction
G    1
Name: count, dtype: int64

Flagged entries:
                    timestamp prediction flag_reason
1  2026-03-01T20:18:42.665288                       



# FSDL Lab 8 Recap — Model Monitoring

## What We Built
Extended our Lab 7 Gradio app with prediction logging and user flagging —
every prediction gets automatically logged, and users can flag wrong
predictions with a reason. Same concept as FSDL's Gantry monitoring demo.

## The Process
1. Added automatic logging — every prediction writes to prediction_log.jsonl
   with timestamp, predicted letter, confidence score
2. Added a Flag button — user types why the prediction was wrong and submits
3. Flagged entries get marked in the log with the reason
4. Built a log analysis cell — shows prediction distribution and all flagged entries

## Key Concepts Practiced
- Automatic prediction logging (timestamp, prediction, confidence)
- User feedback collection via flagging
- JSONL format for append-only logging
- Log analysis with pandas
- Closing the loop: logs → analysis → retraining

## Why Monitoring Matters
A model that works in testing can silently fail in production.
Monitoring catches:
- Distribution shift (users sending unexpected inputs)
- Systematic errors (model always wrong on certain letters)
- Edge cases (low confidence predictions)

Flagging turns users into data labelers — their corrections
become the next round of training data. This is how
production ML systems continuously improve.